# Exercise 6.3: Look-elsewhere effect with toys

From *Programming in High Energy Particle Physics*, Chapter 6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch06/ex06_3_solution.ipynb)

Exercise 6.3 Look-elsewhere effect with toys revised Perform a toy Monte Carlo study of the look-elsewhere effect. Generate 10,000 background-only pseudo-experiments of a smooth mass spectrum over a 100 GeV window, scan each for a Gaussian bump with 1 GeV resolution, and determine how often the largest local fluctuation exceeds 3σ. Compare with the trials-factor estimate of Eq. 5.9 and with the upcrossing formula of Eq. 5.10. Then estimate how many toys you would need to study the 5σ threshold in the same way.

<details><summary>Hint</summary>

For each toy, compute the local significance on a grid of mass hypotheses (for example the counting-experiment significance in a ±2σ window around each mass) and record the maximum. The fraction of toys with maximum above 3σ is \(p_\text{global}\) for \(Z_\text{local} = 3\). Neighboring mass points are correlated, so expect fewer effective trials than grid points. To measure a probability \(p\) with ~10 occurrences you need about \(10/p\) toys.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy scipy matplotlib
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

### 1. Scan background-only spectra

This is a binned counting approximation to a 1 GeV-resolution bump scan. Five-bin windows represent ±2σ. The full requested 10,000 toys are the default; reduce `n_toys` for a quick preview.

In [ ]:
from scipy.stats import poisson, norm
n_toys, n_bins, bg_per_bin = 10_000, 100, 8.0
counts = rng.poisson(bg_per_bin,size=(n_toys,n_bins))
# Count in five-bin windows at each interior mass hypothesis.
cs = np.pad(np.cumsum(counts,axis=1),((0,0),(1,0)))
window = cs[:,5:] - cs[:,:-5]
mu = 5*bg_per_bin
local_p = poisson.sf(window-1,mu)
local_z = norm.isf(np.clip(local_p,1e-300,1-1e-15))
max_z = local_z.max(axis=1)
p_global_3 = np.mean(max_z>=3)
p_local_3 = norm.sf(3)
p_trials_3 = 1-(1-p_local_3)**100
z0=1.0
upcrossings = np.mean(np.sum((local_z[:,:-1]<z0)&(local_z[:,1:]>=z0),axis=1))
p_gv_3 = p_local_3 + upcrossings*np.exp(-(3**2-z0**2)/2)
print(f"toy global p(3σ)={p_global_3:.4f}; N=100 estimate={p_trials_3:.4f}; upcrossing estimate={p_gv_3:.4f}")
print(f"mean upcrossings at Z0=1: {upcrossings:.2f}")
assert 0<p_global_3<0.3

In [ ]:
fig,ax=plt.subplots()
ax.hist(max_z,bins=45,histtype="step",color="navy")
ax.axvline(3,color="crimson",ls="--",label="local 3σ threshold")
ax.set(xlabel="Largest local significance Z",ylabel="Pseudo-experiments / bin")
ax.legend();plt.show()
p_local_5=norm.sf(5)
p_trials_5=1-(1-p_local_5)**100
p_gv_5=p_local_5+upcrossings*np.exp(-(5**2-z0**2)/2)
print(f"local p(5σ)={p_local_5:.2e}; N=100 global estimate={p_trials_5:.2e}")
print(f"upcrossing extrapolation={p_gv_5:.2e}; 10/p estimates: {10/p_trials_5:.0f}, {10/p_gv_5:.0f} toys")

The simulated 3σ global probability should be of order 0.1, though correlated neighboring windows change the effective trials factor. The book’s independent-100-trial estimate is 0.13. The Gross–Vitells estimate uses measured upcrossings at Z₀=1 and is approximate for this discrete Poisson scan. At 5σ, local p≈2.9×10⁻⁷ and the 100-trial estimate is ≈3×10⁻⁵: 10,000 toys yield much less than one case on average, while roughly 10⁵–10⁶ toys are needed for several cases.